In [1]:
%load_ext autoreload
%autoreload 2
%pdb on # clickable err traceback



from __future__ import absolute_import, division, print_function
import torch
from trainer_endoda3 import Trainer
from options_endoda3 import MonodepthOptions


Incorrect argument. Use on/1, off/0, or nothing for a toggle.


/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/third_party/EndoDAC/models/backbones/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/third_party/EndoDAC/models/backbones/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/third_party/EndoDAC/models/backbones/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [9]:
# Minimal options for testing
options = MonodepthOptions()
args = [
    '--batch_size', '2',
    '--num_workers', '1',
    '--of_samples',
    '--of_samples_num', '10',
    '--frame_ids', '0', '-1', '1',
    '--dataset', 'endovis',
    '--data_path', '/mnt/cluster/workspaces/jinjingxu/SCARED_Images_Resized/',
    '--log_dir', '/tmp/endoda_debug',
    '--log_dir', '/mnt/cluster/workspaces/jinjingxu/tmp/',
    '--compute_depth_metrics',
    '--depth_model_type', 'depthanything3',
    '--endoda3_model_config', '/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/networks/configs/endo-da3-all-wowrapper.yaml',
    '--pose_model_type', 'da3_internal',
    '--k_model_type', 'da3_internal',
    '--enable_seq_inputs',
    '--of_supervised_with_which', 'inputs_color',
    '--use_perframe_gt_K',
    '--learn_intrinsics',
    '--train_data_file', 'test_files.txt',
    '--val_data_file', 'test_files.txt',
    '--val_data_file', 'test_files_sequence1_val.txt test_files.txt',
    # '--of_model_type', 'raft',
    # '--use_raft_multi_iters',
    # '--raft_trainable_modules', 'convnormrelu layer1 layer2_0 ',
    # '--learn_intrinsics',
 
    # '--endoda3_model_config', '/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/networks/configs/endo-da3-depth-wowrapper.yaml'
]
opts = options.parse_notebook(args)


# Initialize trainer
trainer = Trainer(opts)
print(f"Trainer initialized on {trainer.device}")


Loading depth model setting from config: /mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/networks/configs/endo-da3-all-wowrapper.yaml
[INFO ] using MLP layer as FFN

Loading pretrained weights from depth-anything/da3-base
[INFO ] using MLP layer as FFN

depth_model state dict info:
  Missing keys: 48
    Key prefixes (up to 3 levels): ['backbone.blocks.0', 'backbone.blocks.1', 'backbone.blocks.10', 'backbone.blocks.11', 'backbone.blocks.2', 'backbone.blocks.3', 'backbone.blocks.4', 'backbone.blocks.5', 'backbone.blocks.6', 'backbone.blocks.7', 'backbone.blocks.8', 'backbone.blocks.9']
  Unexpected keys: 0
Successfully loaded pretrained weights from depth-anything/da3-base for depth net.

Training model named:
   1222_1518
Models and tensorboard events files are saved to:
   /mnt/cluster/workspaces/jinjingxu/tmp/
Training is using:
   cuda
Loading camera intrinsics for 6 unique folders...
Successfully loaded 6 camera intrinsics
Loaded 6 camera intrinsics
  Loaded 10 samples from t

In [3]:
# Get sample batch
trainer.step = 0
trainer.set_train()
train_iter = iter(trainer.train_loader)
inputs = next(train_iter)

# Forward pass
outputs, losses = trainer.process_batch(inputs)

print("Forward pass completed!")
print(f"Output keys: {len(outputs)} keys")
print(f"Loss: {losses['loss'].item():.6f}")
print(f"Loss components: {list(losses.keys())}")


Resizing input from 256x320 to 224x280
[INFO ] Selecting reference view using strategy: first
Forward pass completed!
Output keys: 128 keys
Loss: 0.122331
Loss components: ['loss/0', 'loss/1', 'loss/2', 'loss/3', 'loss']


/mnt/cluster/environments/jinjingxu/pkg/envs/cu12/lib/python3.11/site-packages/torch/nn/functional.py:4902: UserWarning: Default grid_sample and affine_grid behavior has changed to align_corners=False since 1.3.0. Please specify align_corners=True if the old behavior is desired. See the documentation of grid_sample for details.
  warnings.warn(


In [4]:
# Compute losses explicitly
losses = trainer.compute_losses(inputs, outputs)

print("Loss computation:")
for key, val in losses.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key}: {val.item():.6f}")
    else:
        print(f"  {key}: {val}")

# compute losses_0 explicitly
losses_0 = trainer.compute_losses_0(inputs, outputs)

print("Loss computation_0:")
for key, val in losses_0.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key}: {val.item():.6f}")
    else:
        print(f"  {key}: {val}")

Loss computation:
  loss/0: 0.094183
  loss/1: 0.124737
  loss/2: 0.167790
  loss/3: 0.102615
  loss: 0.122331
Loss computation_0:
  loss/0: 0.058496
  loss/1: 0.058495
  loss/2: 0.058494
  loss/3: 0.058495
  loss: 0.058495


In [5]:
# Get validation batch (has GT depth and poses)
trainer.set_eval()
val_iter = iter(trainer.val_loader)
val_inputs = next(val_iter)

# Forward pass
with torch.no_grad():
    val_outputs, val_losses = trainer.process_batch(val_inputs)

# Compute depth metrics
from utils.metrics import compute_depth_metrics, compute_pose_metrics

depth_metrics = compute_depth_metrics(val_inputs, val_outputs)
pose_metrics = compute_pose_metrics(val_inputs, val_outputs, opts.frame_ids)

print("Depth Metrics:")
if depth_metrics:
    for key, val in depth_metrics.items():
        print(f"  {key}: {val:.6f}")
else:
    print("  No depth metrics (GT depth not available)")

print("\nPose Metrics:")
print(pose_metrics)
if pose_metrics:
    for key, val in pose_metrics.items():
        print(f"  {key}: {val:.6f}")
else:
    print("  No pose metrics (GT poses not available)")


Resizing input from 256x320 to 224x280
[INFO ] Selecting reference view using strategy: first
Depth Metrics:
  abs_rel: 0.109665
  sq_rel: 1.266528
  rmse: 9.174189
  rmse_log: 0.159124
  a1: 0.838476
  a2: 0.981557
  a3: 0.999833
  median_scaling_ratio: 489.746307
  median_scaling_std: 0.048356

Pose Metrics:
{}
  No pose metrics (GT poses not available)


In [7]:
# Full training step
trainer.set_train()
train_iter = iter(trainer.train_loader)
inputs = next(train_iter)

# Forward
outputs, losses = trainer.process_batch(inputs)

# Backward
trainer.model_optimizer.zero_grad()
losses["loss"].backward()
trainer.model_optimizer.step()

print(f"Training step completed! Loss: {losses['loss'].item():.6f}")


Resizing input from 256x320 to 224x280
[INFO ] Selecting reference view using strategy: first
Training step completed! Loss: 0.127130
